In [21]:
## Import all the libraries which is required for rag implementation
import os,sys
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_openai.embeddings import OpenAIEmbeddings
from langgraph.graph import START, END, StateGraph
from langchain_core.tools import tool
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.messages import HumanMessage, AIMessage,BaseMessage
from langgraph.prebuilt import ToolNode, tools_condition
from typing import Annotated, TypedDict
from langchain_community.vectorstores import FAISS
from langgraph.graph import add_messages

In [22]:
### Load the environment variables
load_dotenv()

True

In [23]:
model = ChatOpenAI(model="gpt-4o-mini")

In [24]:
### Load the documents from the documents directory
loader = PyPDFLoader("./documuents/[11]part-2-chapter-6.pdf")
docs = loader.load()

In [25]:
len(docs)

60

In [26]:
### text splitters
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
chunks = splitter.split_documents(docs)

len(chunks)

241

In [27]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = FAISS.from_documents(chunks, embeddings)
vector_store

In [28]:
retriever = vector_store.as_retriever(search_type="similarity",search_kwargs={"k":4})

In [29]:
@tool
def rag_tool(query):
    """
    Retrieve relevent information from the pdf documents
    Use this tool when the user asks factual / conceptual questions
    that might be answered from the stored documents
    """
    result = retriever.invoke(query)
    
    content = [doc.page_content for doc in result]
    metadata = [doc.metadata for doc in result]
    
    return {
        "query":query,
        "content":content,
        "metadata":metadata
    }

In [30]:
tools = [rag_tool]
model_with_tools = model.bind_tools(tools)

In [31]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage],add_messages(format="langchain-openai")]

In [33]:
def chat_node(state: ChatState):
    message = state["messages"]
    
    response = model_with_tools.invoke(message)
    
    return {"messages":[response]}

In [34]:
tool_node = ToolNode(tools)

In [40]:
# =========================================
# Build graph
# =========================================

graph = StateGraph(ChatState)

# Add nodes
graph.add_node("chat_node", chat_node)

# IMPORTANT:
# node name must be "tools"
graph.add_node("tools", tool_node)

# Start edge
graph.add_edge(START, "chat_node")

# Conditional edge
graph.add_conditional_edges(
    "chat_node",
    tools_condition
)

# Tool -> chat
graph.add_edge("tools", "chat_node")

# Compile
chatbot = graph.compile()


In [42]:
result = chatbot.invoke({
    "messages":[
        HumanMessage(
            content=(
                "Using the pdf notes, explain what is the feed forward Netork"
            )
        )
    ]
})

In [43]:
print(result["messages"][-1].content)

A feedforward network, also known as a deep feedforward network or multilayer perceptron (MLP), is a fundamental deep learning model. The primary objective of a feedforward network is to approximate a certain function \( f^* \). For instance, in the context of classification, the model maps an input \( x \) to a category \( y \) as \( y = f^*(x) \).

In a feedforward network, the mapping from \( x \) to \( y \) is defined as \( y = f(x; \theta) \), where \( \theta \) represents the parameters of the network that the model learns in order to approximate the desired function effectively. The information in this type of network flows in a singular direction—from the input layer through to the output layer—without any feedback connections. This unidirectional flow of information is what gives the feedforward network its name.

When feedback connections are introduced, the architecture is termed a recurrent neural network, which differs significantly in operation. Feedforward networks are e